*Title*: 8_Genes
*Goal*: All data-related steps for functional/gene stuff.

# 8.1 ID genes of interest  ----------------------
Separating SNPS for each repeatability type. 
``` R
library(tidyverse)
library(data.table)
file<-"repeatability_cleaned_AGAIN.tsv"
data<- read.table(file, header=TRUE, sep="\t")

#Split the text in column 1 to separate chr and pos (separated by _):
data <- data %>%
  separate(ColumnNumber, into = c("chr", "pos"), sep = "_")


#Make local dataframes: (selecting column and position)
local_waddell<-data %>% filter(Local_Waddell == 1) %>% select(1:2) %>% write.table("local_waddell.tsv", sep="\t", row.names=F, col.names = T, quote=F)
local_laguna<-data %>% filter(Local_Laguna == 1) %>% select(1:2) %>% write.table("local_laguna.tsv", sep="\t", row.names=F, col.names = T, quote=F)
local_lombardi <-data %>% filter(Local_Lombardi == 1) %>% select(1:2) %>% write.table("local_lombardi.tsv", sep="\t", row.names=F, col.names = T, quote=F)
local_olddairy<-data %>% filter(Local_OldDairy == 1) %>% select(1:2) %>% write.table("local_olddairy.tsv", sep="\t", row.names=F, col.names = T, quote=F)
local_scott<-data %>% filter(Local_Scott == 1) %>% select(1:2) %>% write.table("local_scott.tsv", sep="\t", row.names=F, col.names = T, quote=F)
local_younger<-data %>% filter(Local_Younger == 1) %>% select(1:2) %>% write.table("local_younger.tsv", sep="\t", row.names=F, col.names = T, quote=F)

#Make high temp_spatial dataframe:
high_spatial_temp<-data %>% filter(spatial_temporal_count == 3) %>% select(1:2) %>% write.table("high_spatial_temp.tsv", sep="\t", row.names=F, col.names = T, quote=F)

#Make 2016_only (high) (4-6 estuaries):
high_only_2016<- data %>% filter(only_2016_spatial %in% c(4, 5, 6)) %>% select(1:2) %>% write.table("high_only_2016.tsv", sep="\t", row.names=F, col.names = T, quote=F)

#Make 2017_only (high) (4-6 estuaries):
high_only_2017<- data %>% filter(only_2017_spatial %in% c(4, 5, 6)) %>% select(1:2) %>% write.table("high_only_2017.tsv", sep="\t", row.names=F, col.names = T, quote=F)
```

 # 8.2 Prepare gff  ----------------------
``` bash
wget https://stickleback.genetics.uga.edu/downloadData/v5_assembly/stickleback_v5_ensembl_genes.gff3.gz

gunzip stickleback_v5_ensembl_genes.gff3.gz
stickleback_v5_ensembl_genes.gff3

grep "gene" stickleback_v5_ensembl_genes.gff3 > stickleback_v5_ensembl_genes_only.gff3 #20805 genes!

grep "protein" stickleback_v5_ensembl_genes.gff3 > stickleback_v5_ensembl_genes_only_protein.gff3 #restrict to protein coding genes. 
```

 # 8.3 Genehits ----------------------
``` bash
#The following genehits.sh script: 
#For the local files:
for file in local*; do  output_file="${file%.tsv}-GO-genehits_ID.tsv"
  while read -r id pos; do
    awk -v id="$id" -v pos="$pos" -f genehits_ID.awk stickleback_v5_ensembl_genes_only_protein.gff3
  done < "$file" > "$output_file"
done

#For the high files:
for file in high*; do
  output_file="${file%.tsv}-GO-genehits_ID.tsv"
  while read -r id pos; do
    awk -v id="$id" -v pos="$pos" -f genehits_ID.awk stickleback_v5_ensembl_genes_only_protein.gff3
  done < "$file" > "$output_file"
done


#genehits.awk script:
function within_range(val, lower, upper, proximity) {
            # you can specify the "proximity" as required
            return val > lower - proximity && val < upper + proximity
        }

        BEGIN {
            OFS="\t"
        }

        $1 == id && within_range(pos, $4, $5, 0) {
            name = gensub(/.*Name=([^\t]*).*/, "\\1", 1)
            if (name ~ /[^[:space:]]+/)
                print id, pos, name
        }
```
 # 8.4 Gene Tables ----------------------
Making tables for the gene lists:
``` R
library(data.table)
library(tidyverse)

#Loop through each file *-GO-genehits_ID.tsv"
files <- list.files(pattern = "-GO-genehits_ID.tsv")

for (file in files) {
  data <- read.table(file, header = FALSE, sep = "\t")
  data <- data %>%
    mutate(V3 = str_replace(V3, "Name=", "")) %>%
    separate(3, into = c("Gene ID", "Gene Name", "protein_id"), sep = ";") %>%
    rename(Chromosome = V1, Position = V2)

#Gene names are duplicated, unlike gene id. Doing everything to match gene id:
  dat <- data %>% select(Chromosome, Position, `Gene ID`,`Gene Name`)
  base_name <- strsplit(basename(file), "-GO")[[1]][1]
  print(base_name)
  print(nrow(dat))
  unique_genes<-unique(dat$`Gene ID`)
  print(length(unique_genes))
  output_file <- paste0(base_name, "_genes_id.tsv")
  write.table(dat, output_file, row.names = FALSE, col.names = TRUE, quote = FALSE, sep = "\t")
}
```
Want to see how which genes have the highest frequency per repeatability type:
``` R
library(tidyverse)
library(data.table)
#Loop through each file *_genes.tsv"
files <- list.files(pattern = "_genes_id.tsv")
data_list <- list()
for (file in files) {
  data <- fread(file, header = TRUE, sep = "\t")
  base_name <- strsplit(basename(file), "_genes")[[1]][1]
  data_list[[base_name]] <- data
}
#now each is saved in data_list
all_gene_names <- lapply(data_list, function(df) df$`Gene ID`)
overlapping_genes <- Reduce(intersect, all_gene_names) #none overalp in all

# Find genes that overlap in the greatest number of datasets (Co-pilot):
# Create a binary presence/absence matrix
gene_presence <- lapply(data_list, function(df) {
  df %>% select(`Gene ID`) %>% distinct() %>% mutate(present = 1)
})

# Combine all presence/absence data into one data frame
combined_presence <- reduce(gene_presence, full_join, by = "Gene ID") %>%
  replace(is.na(.), 0)

# Sum the presence across all files to get the presence count for each gene
file2<-"stickleback_v5_ensembl_genes_only_protein.gff3"
data2<-fread(file2, header=FALSE, sep="\t")
data2 <- data2[,c(1,4,5,9)]
data23<- data2 %>%
mutate(V9 = str_replace(V9, "Name=", "")) %>%
  separate(V9, into = c("Gene ID", "Gene Name", "protein_id"), sep = ";") %>%
  mutate(`Gene ID`= str_replace(`Gene ID`, "ID=", ""))
data23<-data23[,c(1,2,3,4,5)]

gene_presence_count <- combined_presence %>%
  rowwise() %>%
  mutate(presence_count = sum(c_across(starts_with("present")))) %>%
  select(`Gene ID`, presence_count) %>%
  arrange(desc(presence_count)) %>%
  left_join(data23, by = "Gene ID")

# Print the genes that overlap in the greatest number of datasets
print(gene_presence_count)

#Save the gene_presence_count to a file:
write.table(gene_presence_count, "gene_presence_count_with_positions_gene.tsv", sep="\t", row.names=F, col.names = T, quote=F)

#Get the frequencies of each gene in each file, to find the most frequent genes/datast:
frequencies_genes<-data_list %>% map(~count(.x, `Gene ID`)) %>% map(~arrange(., desc(n))) %>% map(~head(., 100))
combined_frequencies_genes <- bind_rows(frequencies_genes, .id = "source") %>%
 left_join(data23, by = "Gene ID") 
write.table(combined_frequencies_genes, "frequencies_genes_per_dataset_with_positions.tsv", sep="\t", row.names=F, col.names = T, quote=F)

#Show gene names unique to each list (copilot):
combined_data <- bind_rows(data_list, .id = "source") #makes a dataset of everything, that shows the source of each gene 
gene_counts <- combined_data %>%
  group_by(`Gene ID`) %>% #this shows the counts of each gene name across the datasets, so if a count of 1, only in 1 dataset (i.e unique to that dataset)
  summarize(count = n_distinct(source))

unique_genes <- combined_data %>%
  inner_join(gene_counts %>% filter(count == 1), by = "Gene ID") #filters each dataset to only show gene names unique to that dataset

frequencies_genes_unique <- split(unique_genes, unique_genes$source) #splits the unique genes into their original datasets
distinct_genes_unique_to_dataset <- lapply(frequencies_genes_unique, function(df) {
  df %>% distinct(`Gene ID`)
})

frequencies_genes_top_unique<-frequencies_genes_unique %>%  map(~count(.x, `Gene ID`)) %>% map(~arrange(., desc(n))) %>% map(~head(., 100))
combined_frequencies_genes_top_unique <- bind_rows(frequencies_genes_top_unique, .id = "source") %>%
  left_join(data23, by = "Gene ID")

#Save:
write.table(combined_frequencies_genes_top_unique, 
            "frequencies_genes_per_dataset_unique_to_each_dataset_with_positions.tsv", 
            sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
```
Making tables for the gene lists:
``` R
#Results summary:
#The number of genes total (all_Gene_names)
# Count the number of entries in each element of the list
num_entries_per_list <- sapply(all_gene_names, length)
total_num_entries <- sum(num_entries_per_list)
print(total_num_entries)


#The number of unique genes in the entire dataset:
# Combine all vectors into one
combined_gene_names <- unlist(all_gene_names)
num_unique_entries <- length(unique(combined_gene_names))
print(num_unique_entries)

#First have to determine the number of unique genes within each dataset (i.e. not repeated gene names in the dataset):
filter_unique_genes <- function(df) {
  df %>% distinct(`Gene ID`, .keep_all = TRUE)
}

# Apply the function to each dataset
unique_genes_per_dataset <- lapply(data_list, filter_unique_genes) 
lengths_unique_genes_per_dataset <- sapply(unique_genes_per_dataset, nrow)

#I want the number of genes that are unique to each dataset (so not present in any other dataset) and also distinct WITHIN that dataset:
lengths_distinct_genes_unique_to_dataset <- sapply(distinct_genes_unique_to_dataset, nrow)

summary_df <- data.frame(
  Dataset = names(num_entries_per_list),
  Total_Entries = num_entries_per_list,
  Unique_Genes = lengths_unique_genes_per_dataset,
  Distinct_Genes = lengths_distinct_genes_unique_to_dataset
)
rownames(summary_df) <- NULL
write.table(summary_df, "summary_gene_counts_by_dataset_gene_ID.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
```
 # 8.5 Gene Figures ----------------------
Graph to show repeatabililty of genes across datasets:
``` R
library(ggplot2)
library(data.table)
library(tidyverse)
library(scales)  
file<-"summary_gene_counts_by_dataset_gene_ID.tsv"
summary_df<-fread(file, header=TRUE, sep="\t")
colnames(summary_df)<-c("Dataset","Total","Non-Distinct","Distinct")

summary_df$Dataset<-recode(summary_df$Dataset,
                                high_only_2016 = "Highly Spatial 2016",
                                high_only_2017 = "Highly Spatial 2017",
                                high_spatial_temp = "Highly\nSpatio-temporal",
                                local_laguna = "Temporal Laguna",
                                local_lombardi = "Temporal Lombardi",
                                local_olddairy = "Temporal Old Dairy",
                                local_scott = "Temporal Scott",
                                local_waddell = "Temporal Waddell",
                                local_younger = "Temporal Younger")

summary_df_mutate <- summary_df[,c(1,3,4)] %>%
mutate(`Non-Distinct`= `Non-Distinct`- Distinct) #I do this to get the counts of unique and non-unique genes(i.e. genes shared with other erpeatability types)
summary_df_long <- pivot_longer(summary_df_mutate, cols = c("Non-Distinct","Distinct"), names_to = "Type", values_to = "value")
summary_df_long$Type <- factor(summary_df_long$Type, levels = c("Non-Distinct","Distinct"))
summary_df_long$original<- summary_df_long$value

#Graph
summary_df_long <- summary_df_long %>% mutate(row_num = row_number()) 

plot <- ggplot(summary_df_long, aes(y = Dataset, x = original, fill = Type)) +
  geom_bar(stat = "identity", position = "stack", color = "black", linewidth = 0.3, width =0.7) +
  theme_classic()+
  theme(axis.text.y = element_text(angle = 0, hjust = 0.5),  # Make x-axis labels horizontal
        panel.grid = element_blank(),
        axis.title.y = element_text(margin = margin(t = 0, r = 15, b = 0, l = 30)),  # Increase left margin for y-axis title
        axis.title.x = element_text(margin = margin(t = 30, r = 0, b = 0, l = 0))) +  # Increase top margin for x-axis title
  labs(y = "Type of Repeatability", x = "Number of Candidate Genes") +
  scale_fill_manual(values = c("Non-Distinct" = "#cccccc", 
                               "Distinct" ="#aaaaaa"),
                               labels = c("Non-Distinct" = "Shared" , 
                               "Distinct" = "Unique"))+
  theme(axis.text.y = element_text(angle = 0, hjust = 0.5),  # Make x-axis labels horizontal
      panel.grid = element_blank()) +  # Remove grid
  geom_text(data = summary_df_long %>% filter(row_num != 6),
  aes(label = paste("n=", comma(original), sep = "")),   
   size = 2.5, color = "black", 
   position = position_stack(vjust = 0.5),  # Adjust vjust to move text above bars
  vjust = -5.5, hjust = 0.5) +      
guides(fill = guide_legend(title = "Type"))

ggsave("stacked_barplot_two_types_not_logged_gene_id_postCM.png", plot = plot, width = 12, height = 10, dpi=700)
ggsave("stacked_barplot_two_types_not_logged_gene_id_postCM.pdf", plot = plot, width = 12, height = 10, dpi=700)
```
 # 8.6 TopGO ----------------------
``` R
library(data.table)
library(topGO)
library(tidyverse)

file<-"high_only_2016-GO-genehits_ID-GO-genehits_ID_filtered.tsv" 
input_file<-fread(file, header=TRUE, sep="\t")
base_name<- sub("-GO-genehits_ID-GO-genehits_ID_filtered.*$", "", basename(file))

gene_pool<-fread("gene_universe_GO_GO_only.tsv", header=TRUE, sep="\t")


# convert from table format to list format
geneID2GO <- by(gene_pool$go_id,
                gene_pool$ensembl_gene_id,
                function(x) as.character(x))

InterestGenes=by(input_file$go_id,
             input_file$ensembl_gene_id,
             function(x) as.character(x))

correction<-"fdr"
geneNames = names(geneID2GO)

myInterestingGenesNames=names(InterestGenes)
geneList = factor(as.integer(geneNames %in% myInterestingGenesNames))
names(geneList) <- geneNames

library(topGO)
ontology=c("MF","BP","CC")

for (i in 1:length(ontology)) {
  tgData = new("topGOdata", 
               ontology = ontology[i], 
               allGenes = geneList, 
               annot = annFUN.gene2GO, 
               gene2GO = geneID2GO,
               nodeSize = 10) #require at least 10 genes in a node
  fisherRes = runTest(tgData, algorithm="classic", statistic="fisher")
  fisherResCor = p.adjust(score(fisherRes), method=correction)
  allRes = GenTable(tgData, 
                    classic=fisherRes, 
                    orderBy="classic", 
                    ranksOf="classic", 
                    topNodes=10)
  allRes$fisher.COR = fisherResCor[allRes$GO.ID]
  output_filename <- paste0(base_name, "_", ontology[i], "_GO_analysis.tsv")
  write.table(allRes, output_filename, row.names = FALSE, col.names = TRUE, quote = FALSE, sep = "\t")
}
#Submitted for each file individually:
high_only_2016-GO-genehits_ID-GO-genehits_ID_filtered.tsv
high_only_2017-GO-genehits_ID-GO-genehits_ID_filtered.tsv
high_spatial_temp-GO-genehits_ID-GO-genehits_ID_filtered.tsv #HAS NOTHING THEREFORE NO RESULTS
local_laguna-GO-genehits_ID-GO-genehits_ID_filtered.tsv
local_lombardi-GO-genehits_ID-GO-genehits_ID_filtered.tsv
local_olddairy-GO-genehits_ID-GO-genehits_ID_filtered.tsv
local_scott-GO-genehits_ID-GO-genehits_ID_filtered.tsv
local_waddell-GO-genehits_ID-GO-genehits_ID_filtered.tsv
local_younger-GO-genehits_ID-GO-genehits_ID_filtered.tsv


#Table to show GO:
library(tidyverse)
library(data.table)
library(xtable)
#Loop through each file *_genes.tsv"
files <- list.files(pattern = "*BP_GO_analysis.tsv")
data_list <- list()
for (file in files) {
  data <- fread(file, header = TRUE, sep = "\t")
  base_name <- strsplit(basename(file), "_BP")[[1]][1]
  data_list[[base_name]] <- data
}

#For each in data_list, filter to only show GO where classic < 0.05
filtered_data_list <- lapply(data_list, function(df) {
  df %>% filter(classic < 0.05) %>%
   head(3)}) #For the supplementary table, I did not head it, but saved all data and exported as .tsv

#Make a dataframe:
combined_data_df <- bind_rows(filtered_data_list, .id = "source") %>%
as.data.frame()

combined_data_df <- combined_data_df %>%
select(1:7) # Select the first 7 columns

colnames(combined_data_df)<-c("Repeatability Type", "GO ID","Term","Annotated","Significant","Expected","Classic p-value")


 combined_data_df <- combined_data_df %>%
 mutate(source = recode(`Repeatability Type`,
                         high_only_2016 = "Highly Spatial 2016",
                         high_only_2017 = "Highly Spatial 2017",
                         high_spatial_temp = "Highly\nSpatio-temporal",
                         local_laguna = "Temporal Laguna",
                         local_lombardi = "Temporal Lombardi",
                         local_olddairy = "Temporal Old Dairy",
                         local_scott = "Temporal Scott",
                         local_waddell = "Temporal Waddell",
                         local_younger = "Temporal Younger")) 


colnames(combined_data_df) <- c("Repeatability Type", "GO ID", "Term", "Annotated", "Significant", "Expected", "Classic p-value", "Source") 

combined_data_df<-combined_data_df[,2:8]
colnames(combined_data_df) <- c("GO ID", "Term", "Annotated", "Significant", "Expected", "Classic p-value", "Repeatability Type") 

combined_data_df <- combined_data_df %>%
  select(`Repeatability Type`, everything())

#Make dataframe a table in tinytex:
xtable_genes <- xtable(combined_data_df,
                       caption = "GO terms for each dataset",
                       align = rep("c", ncol(combined_data_df) + 1),
                       digits = c(0, 0, 0, 0, 0, 0, 4, 5)) # Specify 5 digits for the "Classic p-value" column

print(xtable_genes, 
      type = "latex",
      file = "table_content.tex", 
      include.rownames = FALSE,
      floating = FALSE)

# Create the main LaTeX document
latex_document <- "\\documentclass{article}
\\usepackage{booktabs}
\\usepackage{longtable}
\\usepackage{amssymb}
\\usepackage{geometry}
\\usepackage{rotating}
\\geometry{a4paper, margin=1in}

\\begin{document}
\\begin{sidewaystable}
\\resizebox{\\textwidth}{!}{
\\input{table_content.tex}
}
\\end{sidewaystable}
\\end{document}"

# Write the main LaTeX document
writeLines(latex_document, "genes_OF_BP_postCM.tex")

# Compile to PDF
tinytex::latexmk("genes_OF_BP_postCM.tex")
```